# Milestone 3 Visualization: Parallel Point-in-Polygon Classification
This notebook visualizes the scalability and data distribution patterns of our MPI+OpenMP hybrid solution.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import matplotlib.patches as patches

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.2)

In [ ]:
# 1. Strong Scaling Graph
df = pd.read_csv('benchmark_metrics.csv')
strong = df[df['strategy'].isin(['replicate', 'shard'])]

plt.figure(figsize=(10, 6))
ax = sns.lineplot(data=strong, x='num_nodes', y='exec_time_s', hue='strategy', marker='o', linewidth=2, markersize=8)

# Add ideal linear scaling line based on 1 node
t1_rep = strong[(strong['num_nodes']==1) & (strong['strategy']=='replicate')]['exec_time_s'].values[0]
nodes = sorted(strong['num_nodes'].unique())
ideal_time = [t1_rep / n for n in nodes]
plt.plot(nodes, ideal_time, 'k--', label='Ideal Linear Scaling', alpha=0.7)

plt.title('Strong Scaling (Fixed 100M Points)', fontsize=16, fontweight='bold')
plt.xlabel('Number of MPI Nodes', fontsize=14)
plt.ylabel('Execution Time (seconds)', fontsize=14)
plt.xticks(nodes)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(title='Strategy', fontsize=12, title_fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Weak Scaling Graph
weak = df[df['strategy'].isin(['replicate_weak', 'shard_weak'])]
if not weak.empty:
    plt.figure(figsize=(10, 6))
    sns.lineplot(data=weak, x='num_nodes', y='exec_time_s', hue='strategy', marker='s', linewidth=2, markersize=8)
    
    # Ideal weak scaling is constant time
    t1_weak = weak[(weak['num_nodes']==1) & (weak['strategy']=='replicate_weak')]['exec_time_s'].values[0]
    plt.axhline(y=t1_weak, color='k', linestyle='--', label='Ideal Weak Scaling', alpha=0.7)
    
    plt.title('Weak Scaling (10M Points per Node)', fontsize=16, fontweight='bold')
    plt.xlabel('Number of MPI Nodes', fontsize=14)
    plt.ylabel('Execution Time (seconds)', fontsize=14)
    plt.xticks(nodes)
    plt.ylim(bottom=0)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(title='Strategy', fontsize=12, title_fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("No weak scaling data available in benchmark_metrics.csv. Make sure to run weak scaling experiments.")

In [ ]:
# 3. Spatial Skew Heatmap
skew_df = pd.read_csv('spatial_skew.csv', header=None)

plt.figure(figsize=(10, 8))
sns.heatmap(skew_df, cmap='inferno', square=True, cbar_kws={'label': 'Point Density'})
plt.title('Spatial Skew: Point Density Heatmap', fontsize=16, fontweight='bold')
plt.xlabel('Grid X', fontsize=14)
plt.ylabel('Grid Y', fontsize=14)
plt.gca().invert_yaxis() # Match standard coordinate system where 0,0 is bottom left
plt.tight_layout()
plt.show()

In [ ]:
# 4. MPI Domain Decomposition Map
decomp_df = pd.read_csv('domain_decomposition.csv')

fig, ax = plt.subplots(figsize=(10, 10))

colors = sns.color_palette("Set2", n_colors=decomp_df['assigned_rank'].nunique())
color_map = {rank: color for rank, color in zip(sorted(decomp_df['assigned_rank'].unique()), colors)}

# Plot each polygon's bounding box
for _, row in decomp_df.iterrows():
    width = row['bbox_maxX'] - row['bbox_minX']
    height = row['bbox_maxY'] - row['bbox_minY']
    rect = patches.Rectangle((row['bbox_minX'], row['bbox_minY']), width, height,
                             linewidth=1.5, edgecolor=color_map[row['assigned_rank']],
                             facecolor=color_map[row['assigned_rank']], alpha=0.4)
    ax.add_patch(rect)

# Draw grid lines for partitions
ax.set_xlim(0, 100) # Assuming 100x100 domain based on dataset
ax.set_ylim(0, 100)

# Create custom legend
legend_patches = [patches.Patch(color=color_map[rank], label=f'Rank {rank}') 
                  for rank in sorted(color_map.keys())]
ax.legend(handles=legend_patches, title='Assigned MPI Rank', loc='upper right', bbox_to_anchor=(1.2, 1))

plt.title('MPI Spatial Sharding: Domain Decomposition', fontsize=16, fontweight='bold')
plt.xlabel('X Coordinate', fontsize=14)
plt.ylabel('Y Coordinate', fontsize=14)
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()